In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

def build_professional_denoiser(input_shape=(256, 256, 3)):
    """
    Arsitektur Denoising berbasis U-Net.
    Dirancang untuk mempertahankan detail spasial (edges/textures) 
    melalui penggunaan skip connections.
    """
    inputs = layers.Input(shape=input_shape, name="noisy_image_input")

    # --- ENCODER (Downsampling) ---
    # Mengekstrak fitur hierarkis.
    conv1 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(inputs)
    conv1 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(conv1)
    pool1 = layers.MaxPooling2D((2, 2))(conv1)

    conv2 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(pool1)
    conv2 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(conv2)
    pool2 = layers.MaxPooling2D((2, 2))(conv2)

    # --- BOTTLENECK ---
    conv3 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(pool2)
    conv3 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(conv3)

    # --- DECODER (Upsampling) ---
    # Merekonstruksi resolusi sambil menggabungkan fitur detail dari Encoder.
    up4 = layers.Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(conv3)
    merge4 = layers.concatenate([up4, conv2])
    conv4 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(merge4)
    conv4 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(conv4)

    up5 = layers.Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(conv4)
    merge5 = layers.concatenate([up5, conv1])
    conv5 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(merge5)
    conv5 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(conv5)

    # Output layer: 3 channels (RGB)
    outputs = layers.Conv2D(3, (1, 1), activation='sigmoid', name="clean_image_output")(conv5)

    model = Model(inputs=inputs, outputs=outputs, name="Professional_Denoiser")
    return model

# Inisialisasi model
model = build_professional_denoiser()

# Sesuai teori, kita memerlukan loss function untuk menghitung jarak ke ground truth
# Dokumen mendefinisikan L = (y - y^)^2 (Mean Squared Error).
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
              loss='mse',
              metrics=['mae'])

model.summary()

Model: "Professional_Denoiser"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ noisy_image_input   │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 256, 256,  │      1,792 │ noisy_image_inpu… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 256, 256,  │     36,928 │ conv2d[0][0]      │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 128, 128,  │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 128, 128,  │     73,856 │ max_pooling2d[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 128, 128,  │    147,584 │ conv2d_2[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 64, 64,    │          0 │ conv2d_3[0][0]    │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 64, 64,    │    295,168 │ max_pooling2d_1[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 64, 64,    │    590,080 │ conv2d_4[0][0]    │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose    │ (None, 128, 128,  │    131,200 │ conv2d_5[0][0]    │
│ (Conv2DTranspose)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 128, 128,  │          0 │ conv2d_transpose… │
│ (Concatenate)       │ 256)              │            │ conv2d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 128, 128,  │    295,040 │ concatenate[0][0] │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 128, 128,  │    147,584 │ conv2d_6[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose_1  │ (None, 256, 256,  │     32,832 │ conv2d_7[0][0]    │
│ (Conv2DTranspose)   │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 256, 256,  │          0 │ conv2d_transpose… │
│ (Concatenate)       │ 128)              │            │ conv2d_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, 256, 256,  │     73,792 │ concatenate_1[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_9 (Conv2D)   │ (None, 256, 256,  │     36,928 │ conv2d_8[0][0]  

 Total params: 1,862,979 (7.11 MB)

 Trainable params: 1,862,979 (7.11 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
# ============================================================
# DENOISING DENGAN NEURAL NETWORK
# Berdasarkan materi UCSD CSE 167 - Neural Networks (Tzu-Mao Li)
# ============================================================

# ─────────────────────────────────────────────────────────────
# CELL 1: Import Library
# ─────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

print("✅ Semua library berhasil diimport!")
print(f"PyTorch version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device yang digunakan: {device}")


# ─────────────────────────────────────────────────────────────
# CELL 2: Memahami Konsep Noise
# ─────────────────────────────────────────────────────────────
"""
PENJELASAN KONSEP NOISE (dari slide CSE 167):

Path tracing menghasilkan gambar dengan sampling acak cahaya.
Semakin SEDIKIT samples per pixel (spp), gambar semakin NOISY.

Contoh dari slide:
  - 8 spp    → sangat noisy
  - 40 spp   → masih noisy
  - 200 spp  → cukup bersih
  - 1000 spp → bersih, tapi LAMBAT dan butuh banyak komputasi

SOLUSI: Gunakan neural network untuk "menebak" gambar bersih
dari gambar noisy dengan spp rendah!
"""

# Visualisasi efek noise pada gambar
def tambah_noise(gambar, sigma=0.1):
    """Tambahkan Gaussian noise ke gambar. sigma = tingkat kebisingan."""
    noise = np.random.normal(0, sigma, gambar.shape)
    gambar_noisy = np.clip(gambar + noise, 0, 1)
    return gambar_noisy

# Buat gambar sederhana untuk demo
np.random.seed(42)
gambar_bersih = np.zeros((64, 64, 3))
gambar_bersih[10:30, 10:30] = [1, 0, 0]   # kotak merah
gambar_bersih[10:30, 35:55] = [0, 1, 0]   # kotak hijau
gambar_bersih[35:55, 22:42] = [0, 0, 1]   # kotak biru

# Simulasi berbagai level noise (seperti samples per pixel)
noise_levels = [0.05, 0.15, 0.3, 0.5]
fig, axes = plt.subplots(1, 5, figsize=(16, 4))
axes[0].imshow(gambar_bersih)
axes[0].set_title("Gambar Bersih\n(1000 spp)", fontsize=11)
axes[0].axis('off')

for i, sigma in enumerate(noise_levels):
    noisy = tambah_noise(gambar_bersih, sigma)
    axes[i+1].imshow(noisy)
    axes[i+1].set_title(f"σ = {sigma}\n(simulasi {['200','40','20','8'][i]} spp)", fontsize=11)
    axes[i+1].axis('off')

plt.suptitle("Efek Noise pada Gambar (semakin tinggi σ = semakin noisy)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/01_noise_demo.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Visualisasi noise berhasil dibuat!")


# ─────────────────────────────────────────────────────────────
# CELL 3: Dataset - Pasangan Gambar Noisy/Bersih
# ─────────────────────────────────────────────────────────────
"""
IDE UTAMA DARI SLIDE:
"Train neural networks on noisy/clean image pairs"

Artinya: kita siapkan dataset berisi:
  INPUT  → gambar noisy (sedikit samples per pixel)
  TARGET → gambar bersih (banyak samples per pixel)
  
Neural network belajar memetakan: noisy → bersih
"""

class NoisyDataset(Dataset):
    """
    Dataset yang secara otomatis membuat pasangan (noisy, clean).
    Gunakan gambar MNIST sebagai contoh.
    """
    def __init__(self, train=True, sigma=0.2):
        self.sigma = sigma
        transform = transforms.Compose([
            transforms.ToTensor(),
        ])
        self.data = datasets.MNIST(
            root='./data', train=train, download=True, transform=transform
        )
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        gambar_bersih, _ = self.data[idx]  # ambil gambar (abaikan label)
        noise = torch.randn_like(gambar_bersih) * self.sigma
        gambar_noisy = torch.clamp(gambar_bersih + noise, 0, 1)
        return gambar_noisy, gambar_bersih  # input=noisy, target=bersih

# Buat dataset
print("📦 Memuat dataset MNIST...")
train_dataset = NoisyDataset(train=True, sigma=0.3)
test_dataset  = NoisyDataset(train=False, sigma=0.3)
train_loader  = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader   = DataLoader(test_dataset, batch_size=64, shuffle=False)
print(f"✅ Training samples : {len(train_dataset):,}")
print(f"✅ Testing samples  : {len(test_dataset):,}")

# Visualisasi beberapa sampel dataset
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
noisy_batch, clean_batch = next(iter(test_loader))

for i in range(6):
    axes[0, i].imshow(noisy_batch[i].squeeze(), cmap='gray')
    axes[0, i].set_title("Noisy (Input)", fontsize=9)
    axes[0, i].axis('off')
    axes[1, i].imshow(clean_batch[i].squeeze(), cmap='gray')
    axes[1, i].set_title("Bersih (Target)", fontsize=9)
    axes[1, i].axis('off')

plt.suptitle("Contoh Pasangan Data: Noisy vs Clean", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/02_dataset_sample.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Visualisasi dataset berhasil!")


# ─────────────────────────────────────────────────────────────
# CELL 4: Arsitektur Neural Network - Denoising CNN
# ─────────────────────────────────────────────────────────────
"""
ARSITEKTUR YANG DIGUNAKAN: DnCNN (Denoising CNN)

Lapisan-lapisan (dari slide):
1. Convolution  → setiap pixel output = kombinasi pixel tetangganya
                  (bisa "mengenal" pola noise)
2. Activation   → ReLU: y = max(x, 0) - menambah non-linearitas
3. Batch Norm   → stabilkan training
4. Convolution  → lapisan lebih dalam = fitur lebih kompleks
...
5. Output Conv  → hasilkan gambar bersih

Trik: Residual Learning
  daripada prediksi gambar bersih langsung,
  neural net belajar prediksi NOISE-nya,
  lalu: gambar_bersih = gambar_noisy - prediksi_noise
"""

class DnCNN(nn.Module):
    """
    Denoising Convolutional Neural Network.
    Belajar memprediksi noise, bukan gambar bersihnya langsung.
    """
    def __init__(self, channels=1, num_layers=6, features=32):
        super(DnCNN, self).__init__()
        lapisan = []
        
        # Lapisan pertama: Conv + ReLU
        lapisan.append(nn.Conv2d(channels, features, kernel_size=3, padding=1, bias=False))
        lapisan.append(nn.ReLU(inplace=True))
        
        # Lapisan tengah: Conv + BatchNorm + ReLU
        for _ in range(num_layers - 2):
            lapisan.append(nn.Conv2d(features, features, kernel_size=3, padding=1, bias=False))
            lapisan.append(nn.BatchNorm2d(features))
            lapisan.append(nn.ReLU(inplace=True))
        
        # Lapisan terakhir: Conv (output = prediksi noise)
        lapisan.append(nn.Conv2d(features, channels, kernel_size=3, padding=1, bias=False))
        
        self.network = nn.Sequential(*lapisan)
    
    def forward(self, x):
        # Residual learning: prediksi noise, lalu kurangkan
        noise_pred = self.network(x)
        return x - noise_pred  # gambar_bersih = noisy - noise_yang_diprediksi


# Buat model dan tampilkan info
model = DnCNN(channels=1, num_layers=6, features=32).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"📐 Arsitektur Model:")
print(model)
print(f"\n📊 Total parameter: {total_params:,}")
print(f"✅ Model siap di-train di {device}")


# ─────────────────────────────────────────────────────────────
# CELL 5: Training Loop
# ─────────────────────────────────────────────────────────────
"""
PROSES TRAINING (dari slide CSE 167):

1. Feed input (gambar noisy) ke neural network → f(x, p)
2. Hitung loss: L = (y - ŷ)²   [Mean Squared Error]
   y  = gambar bersih (target)
   ŷ  = output neural network
3. Adjust parameter: p ← p - λ (∂L/∂p)
   λ = learning rate
   ∂L/∂p = gradien (otomatis dihitung PyTorch)
4. Ulangi untuk semua data (1 epoch)
"""

def train_satu_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for noisy, bersih in loader:
        noisy, bersih = noisy.to(device), bersih.to(device)
        
        optimizer.zero_grad()          # reset gradien
        output = model(noisy)          # forward pass
        loss = criterion(output, bersih)  # hitung loss
        loss.backward()               # backward pass (hitung ∂L/∂p)
        optimizer.step()              # update parameter
        
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluasi(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    psnr_total = 0
    n_batch = 0
    with torch.no_grad():
        for noisy, bersih in loader:
            noisy, bersih = noisy.to(device), bersih.to(device)
            output = model(noisy)
            loss = criterion(output, bersih)
            total_loss += loss.item()
            
            # Hitung PSNR (Peak Signal-to-Noise Ratio)
            # Semakin tinggi PSNR, semakin mirip dengan gambar asli
            out_np = output.cpu().numpy().squeeze()
            bersih_np = bersih.cpu().numpy().squeeze()
            if out_np.ndim == 3:
                out_np = out_np[0]
                bersih_np = bersih_np[0]
            psnr_val = psnr(bersih_np, np.clip(out_np, 0, 1), data_range=1.0)
            psnr_total += psnr_val
            n_batch += 1
    
    return total_loss / len(loader), psnr_total / n_batch

# Setup optimizer dan loss
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

# Training
N_EPOCH = 10
history = {'train_loss': [], 'val_loss': [], 'val_psnr': []}

print("🚀 Mulai training...\n")
print(f"{'Epoch':>6} | {'Train Loss':>12} | {'Val Loss':>10} | {'PSNR (dB)':>10}")
print("-" * 50)

for epoch in range(1, N_EPOCH + 1):
    train_loss = train_satu_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_psnr = evaluasi(model, test_loader, criterion, device)
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_psnr'].append(val_psnr)
    
    print(f"{epoch:>6} | {train_loss:>12.6f} | {val_loss:>10.6f} | {val_psnr:>10.2f}")

print("\n✅ Training selesai!")


# ─────────────────────────────────────────────────────────────
# CELL 6: Plot Kurva Training
# ─────────────────────────────────────────────────────────────

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_loss'], 'b-o', label='Training Loss', linewidth=2, markersize=5)
ax1.plot(history['val_loss'],   'r-o', label='Validation Loss', linewidth=2, markersize=5)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('MSE Loss', fontsize=12)
ax1.set_title('Training vs Validation Loss\n(semakin turun = semakin bagus)', fontsize=12)
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history['val_psnr'], 'g-o', linewidth=2, markersize=5)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('PSNR (dB)', fontsize=12)
ax2.set_title('PSNR Validation\n(semakin tinggi = gambar semakin bersih)', fontsize=12)
ax2.grid(True, alpha=0.3)

plt.suptitle("Kurva Training Neural Network Denoiser", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/03_training_curve.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Kurva training berhasil diplot!")


# ─────────────────────────────────────────────────────────────
# CELL 7: Visualisasi Hasil Denoising
# ─────────────────────────────────────────────────────────────

model.eval()
noisy_batch, clean_batch = next(iter(test_loader))
with torch.no_grad():
    output_batch = model(noisy_batch.to(device)).cpu()

output_batch = torch.clamp(output_batch, 0, 1)

fig, axes = plt.subplots(3, 7, figsize=(16, 7))
titles = ["Noisy (Input)", "Output Neural Net", "Bersih (Target)"]

for i in range(7):
    noisy_img  = noisy_batch[i].squeeze().numpy()
    output_img = output_batch[i].squeeze().numpy()
    clean_img  = clean_batch[i].squeeze().numpy()
    
    p_noisy  = psnr(clean_img, noisy_img, data_range=1.0)
    p_output = psnr(clean_img, output_img, data_range=1.0)
    
    axes[0, i].imshow(noisy_img,  cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(f"PSNR: {p_noisy:.1f} dB", fontsize=8)
    axes[0, i].axis('off')
    
    axes[1, i].imshow(output_img, cmap='gray', vmin=0, vmax=1)
    axes[1, i].set_title(f"PSNR: {p_output:.1f} dB", fontsize=8)
    axes[1, i].axis('off')
    
    axes[2, i].imshow(clean_img,  cmap='gray', vmin=0, vmax=1)
    axes[2, i].set_title("Referensi", fontsize=8)
    axes[2, i].axis('off')

for ax, title in zip(axes[:, 0], titles):
    ax.set_ylabel(title, fontsize=10, fontweight='bold', rotation=90, va='center')

plt.suptitle("Hasil Denoising: Noisy → Neural Net → Bersih\n(PSNR lebih tinggi = lebih baik)", 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/04_denoising_result.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Visualisasi hasil denoising berhasil!")


# ─────────────────────────────────────────────────────────────
# CELL 8: Perbandingan Berbagai Level Noise
# ─────────────────────────────────────────────────────────────
"""
Sesuai slide: lebih banyak samples per pixel = gambar lebih bersih.
Di sini kita simulasikan dengan berbagai level sigma noise.
"""

model.eval()
sample_img, _ = test_dataset[0]
sample_img = sample_img.unsqueeze(0)

sigma_levels = [0.05, 0.15, 0.3, 0.5]
fig, axes = plt.subplots(3, len(sigma_levels)+1, figsize=(16, 8))

# Kolom pertama: gambar bersih
axes[0, 0].imshow(sample_img.squeeze(), cmap='gray')
axes[0, 0].set_title("Bersih (Asli)", fontsize=10)
axes[0, 0].axis('off')
axes[1, 0].axis('off')
axes[2, 0].axis('off')

labels_row = ["Gambar Noisy", "Hasil Denoising", "Selisih (Error)"]

for j, sigma in enumerate(sigma_levels):
    noise = torch.randn_like(sample_img) * sigma
    noisy_img = torch.clamp(sample_img + noise, 0, 1)
    
    with torch.no_grad():
        output = torch.clamp(model(noisy_img.to(device)).cpu(), 0, 1)
    
    diff = torch.abs(output - sample_img)
    
    p_in  = psnr(sample_img.squeeze().numpy(), noisy_img.squeeze().numpy(), data_range=1.0)
    p_out = psnr(sample_img.squeeze().numpy(), output.squeeze().numpy(),    data_range=1.0)
    
    axes[0, j+1].imshow(noisy_img.squeeze(), cmap='gray')
    axes[0, j+1].set_title(f"σ={sigma}\nPSNR: {p_in:.1f}dB", fontsize=9)
    axes[0, j+1].axis('off')
    
    axes[1, j+1].imshow(output.squeeze(), cmap='gray')
    axes[1, j+1].set_title(f"Setelah Denoise\nPSNR: {p_out:.1f}dB", fontsize=9)
    axes[1, j+1].axis('off')
    
    axes[2, j+1].imshow(diff.squeeze(), cmap='hot')
    axes[2, j+1].set_title("Error Map", fontsize=9)
    axes[2, j+1].axis('off')

for ax, label in zip(axes[:, 0], labels_row):
    ax.text(0.5, 0.5, label, transform=ax.transAxes, fontsize=10,
            fontweight='bold', ha='center', va='center')

plt.suptitle("Denoising pada Berbagai Level Noise (σ)\n(Simulasi jumlah samples per pixel berbeda)",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/05_noise_level_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Perbandingan level noise berhasil!")


# ─────────────────────────────────────────────────────────────
# CELL 9: Simpan dan Muat Model
# ─────────────────────────────────────────────────────────────

# Simpan model
torch.save(model.state_dict(), '/tmp/denoising_model.pth')
print("💾 Model tersimpan di: /tmp/denoising_model.pth")

# Cara memuat ulang model
model_baru = DnCNN(channels=1, num_layers=6, features=32).to(device)
model_baru.load_state_dict(torch.load('/tmp/denoising_model.pth', map_location=device))
model_baru.eval()
print("✅ Model berhasil dimuat ulang!")


# ─────────────────────────────────────────────────────────────
# CELL 10: Ringkasan & Kesimpulan
# ─────────────────────────────────────────────────────────────

print("""
╔══════════════════════════════════════════════════════════════╗
║           RINGKASAN: DENOISING DENGAN NEURAL NETWORK        ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  KONSEP UTAMA (dari slide CSE 167):                         ║
║  • Path tracing menghasilkan noise saat spp rendah           ║
║  • Neural network dilatih pada pasangan noisy/clean          ║
║  • Setelah training, bisa "membersihkan" gambar noisy        ║
║                                                              ║
║  ARSITEKTUR (lapisan dari slide):                            ║
║  Convolution → BatchNorm → ReLU (× N layer)                 ║
║  + Residual Learning (prediksi noise, bukan gambar bersih)  ║
║                                                              ║
║  PROSES TRAINING:                                            ║
║  1. Input: gambar noisy → f(x, p)                           ║
║  2. Loss: L = (y - ŷ)²                                      ║
║  3. Update: p ← p - λ (∂L/∂p)                              ║
║                                                              ║
║  METRIK EVALUASI:                                            ║
║  • MSE Loss: semakin kecil = semakin baik                   ║
║  • PSNR (dB): semakin tinggi = semakin bersih               ║
║  • SSIM: semakin mendekati 1 = semakin mirip asli           ║
║                                                              ║
║  APLIKASI NYATA (disebutkan di slide):                       ║
║  • Denoising render 3D (NVIDIA OptiX Denoiser)              ║
║  • Kamera smartphone                                         ║
║  • Komponen wajib renderer modern!                           ║
╚══════════════════════════════════════════════════════════════╝
""")

RuntimeError: operator torchvision::nms does not exist

In [8]:
!pip uninstall torch torchvision torchaudio -y

Found existing installation: torch 2.12.0
Uninstalling torch-2.12.0:
  Successfully uninstalled torch-2.12.0
Found existing installation: torchvision 0.27.0
Uninstalling torchvision-0.27.0:
  Successfully uninstalled torchvision-0.27.0


In [11]:
!pip cache purge

Files removed: 2084 (1681.5 MB)


In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu --force-reinstall --no-cache-dir

Looking in indexes: https://download.pytorch.org/whl/cpu
   ---------------------------------------- 0.0/122.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/122.9 MB ? eta -:--:--
   ---------------------------------------- 0.8/122.9 MB 2.8 MB/s eta 0:00:44
   ---------------------------------------- 1.3/122.9 MB 2.2 MB/s eta 0:00:55
    --------------------------------------- 1.8/122.9 MB 2.4 MB/s eta 0:00:51
    --------------------------------------- 2.6/122.9 MB 2.6 MB/s eta 0:00:47
   - -------------------------------------- 3.1/122.9 MB 2.7 MB/s eta 0:00:45
   - -------------------------------------- 3.7/122.9 MB 2.7 MB/s eta 0:00:45
   - -------------------------------------- 4.2/122.9 MB 2.7 MB/s eta 0:00:45
   - -------------------------------------- 4.5/122.9 MB 2.6 MB/s eta 0:00:46
   - -------------------------------------- 4.5/122.9 MB 2.6 MB/s eta 0:00:46
   - -------------------------------------- 5.0/122.9 MB 2.3 MB/s eta 0:00:52
   - ----------------

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cleanrl 1.2.0 requires huggingface-hub<0.12.0,>=0.11.1, but you have huggingface-hub 0.36.0 which is incompatible.
cleanrl 1.2.0 requires stable-baselines3==1.2.0, but you have stable-baselines3 2.7.1 which is incompatible.
cleanrl 1.2.0 requires wandb<0.14.0,>=0.13.11, but you have wandb 0.23.1 which is incompatible.
datasets 4.4.1 requires fsspec[http]<=2025.10.0,>=2023.1.0, but you have fsspec 2026.4.0 which is incompatible.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.2.6 which is incompatible.
typer-slim 0.24.0 requires typer>=0.24.0, but you have typer 0.12.5 which is incompatible.


: 

In [10]:
import torch
import torchvision
print("Torch:", torch.__version__)
print("Vision:", torchvision.__version__)

RuntimeError: operator torchvision::nms does not exist